In [1]:
import pandas as pd
import numpy as np
import re
import joblib
import scipy.sparse as sp
from pathlib import Path
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from xgboost import XGBClassifier

BASE_DIR  = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "cleaned_resumes.csv"
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(exist_ok=True)

In [2]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df['target'].value_counts())
print(df.columns)

# Safety
df = df.dropna(subset=['cleaned_resume'])
df = df[df['cleaned_resume'].str.strip() != '']

Shape: (2483, 9)
target
0    2245
1     238
Name: count, dtype: int64
Index(['ID', 'Resume_str', 'Category', 'cleaned_resume', 'target',
       'years_experience', 'has_degree', 'skill_keywords', 'embed_resume'],
      dtype='object')


In [3]:
TECH_SKILLS = [
    'python','java','c++','c','javascript','typescript','r',
    'machine learning','deep learning','nlp','computer vision',
    'tensorflow','pytorch','scikit-learn','pandas','numpy',
    'node','react','angular','django','flask','spring','api','rest',
    'aws','azure','gcp','docker','kubernetes','linux',
    'sql','mysql','postgresql','mongodb','spark','hadoop',
    'git','ci/cd','devops','data science','ai'
]

def extract_years(text):
    text = str(text).lower()

    word_to_num = {
        "one":1,"two":2,"three":3,"four":4,"five":5,
        "six":6,"seven":7,"eight":8,"nine":9,"ten":10
    }

    years = 0

    matches = re.findall(r'(\d+)\s*(year|yr)', text)
    if matches:
        years = max(int(m[0]) for m in matches)
    else:
        for word, num in word_to_num.items():
            if f"{word} year" in text or f"{word} yr" in text:
                years = num
                break

    return years


def has_degree(text):
    text = str(text).lower()
    keywords = ['bachelor','master','phd','mba','b.tech','m.tech','degree']
    return int(any(k in text for k in keywords))


def skill_count(text):
    text = str(text).lower()
    count = 0
    for skill in TECH_SKILLS:
        if re.search(rf"\b{re.escape(skill)}\b", text):
            count += 1
    return count


df['years_experience'] = df['cleaned_resume'].apply(extract_years)
df['has_degree']       = df['cleaned_resume'].apply(has_degree)
df['skill_keywords']   = df['cleaned_resume'].apply(skill_count)

print(df[['years_experience','has_degree','skill_keywords']].head())

   years_experience  has_degree  skill_keywords
0                 0           0               0
1                 0           1               0
2                20           1               0
3                20           0               0
4                 0           1               0


In [4]:
tfidf = TfidfVectorizer(
    max_features=1500,   
    ngram_range=(1,2),
    stop_words='english',
    min_df=2,
    max_df=0.85
)

X_text  = df['cleaned_resume']
X_tfidf = tfidf.fit_transform(X_text)

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (2483, 1500)


In [5]:
X_extra = df[['years_experience','has_degree','skill_keywords']].values

scaler = StandardScaler()
X_extra_scaled = scaler.fit_transform(X_extra)

X = sp.hstack([X_tfidf, sp.csr_matrix(X_extra_scaled)])

y = df['target'].values

print("Final feature shape:", X.shape)

Final feature shape: (2483, 1503)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [7]:
neg, pos = np.bincount(y_train)
scale_pw = neg / pos

print("scale_pos_weight:", scale_pw)

scale_pos_weight: 9.452631578947368


In [8]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [9]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("Proba range:", y_proba.min(), "→", y_proba.max())

              precision    recall  f1-score   support

           0       0.98      0.97      0.97       449
           1       0.72      0.79      0.75        48

    accuracy                           0.95       497
   macro avg       0.85      0.88      0.86       497
weighted avg       0.95      0.95      0.95       497

ROC-AUC: 0.9749907201187824
Proba range: 2.922847e-05 → 0.99940443


In [10]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]

print("Best threshold:", best_thresh)
print("Best F1:", f1_scores[best_idx])

Best threshold: 0.45823383
Best F1: 0.7809523759891158


In [11]:
joblib.dump(model, MODEL_DIR / "model.pkl")
joblib.dump(tfidf, MODEL_DIR / "vectorizer.pkl")
joblib.dump(tfidf.get_feature_names_out(), MODEL_DIR / "feature_names.pkl")
joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
joblib.dump(best_thresh, MODEL_DIR / "threshold.pkl")

print("All artifacts saved to:", MODEL_DIR)

All artifacts saved to: c:\HireIQ\models
